# Cart2Insights - SQL Business Analysis

This notebook demonstrates SQL-based business analysis using the
Cart2Insights MySQL database.

In [1]:
import os
import pandas as pd

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

In [2]:
load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

password = quote_plus(DB_PASSWORD)

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{password}@"
    f"{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created successfully.")

c:\Users\Zoya khan\AppData\Local\Programs\Python\Python37\lib\site-packages\pymysql\_auth.py:9: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  from cryptography.hazmat.backends import default_backend


Database connection created successfully.


In [3]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT 1")
    )

print("MySQL connection successful.")

MySQL connection successful.


SELECT

In [4]:
query = """
SELECT
    order_id,
    order_status,
    order_purchase_timestamp
FROM orders
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,order_id,order_status,order_purchase_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02
1,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06
2,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31
3,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51
5,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,2017-05-15 21:42:34
6,00054e8431b9d7675808bcb819fb4a32,delivered,2017-12-10 11:53:48
7,000576fe39319847cbb9d288c5617fa6,delivered,2018-07-04 12:08:27
8,0005a1a1728c9d785b8e2b08b904576c,delivered,2018-03-19 18:40:33
9,0005f50442cb953dcd1d21e1fb923495,delivered,2018-07-02 13:59:39


WHERE


In [6]:
query = """
SELECT
    order_id,
    order_status,
    order_purchase_timestamp
FROM orders
WHERE order_status = 'delivered'
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,order_id,order_status,order_purchase_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02
1,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06
2,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31
3,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51
5,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,2017-05-15 21:42:34
6,00054e8431b9d7675808bcb819fb4a32,delivered,2017-12-10 11:53:48
7,000576fe39319847cbb9d288c5617fa6,delivered,2018-07-04 12:08:27
8,0005a1a1728c9d785b8e2b08b904576c,delivered,2018-03-19 18:40:33
9,0005f50442cb953dcd1d21e1fb923495,delivered,2018-07-02 13:59:39


ORDER BY

In [7]:
query = """
SELECT
    order_id,
    order_purchase_timestamp
FROM orders
ORDER BY order_purchase_timestamp DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,order_id,order_purchase_timestamp
0,10a045cdf6a5650c21e9cfeb60384c16,2018-10-17 17:30:18
1,b059ee4de278302d550a3035c4cdb740,2018-10-16 20:16:02
2,a2ac6dad85cf8af5b0afb510a240fe8c,2018-10-03 18:55:29
3,616fa7d4871b87832197b2a137a115d2,2018-10-01 15:30:09
4,392ed9afd714e3c74767d0c4d3e3f477,2018-09-29 09:13:03
5,869997fbe01f39d184956b5c6bccfdbe,2018-09-26 08:40:15
6,5aac76cf7b07dd06fa4d50bf461d2f40,2018-09-25 11:59:18
7,ed3efbd3a87bea76c2812c66a0b32219,2018-09-20 13:54:16
8,bd35b677fd239386e9861d11ae98ab56,2018-09-17 17:21:16
9,ea844c92cf978ea23321fa7fe5871761,2018-09-13 09:56:12


GROUP BY + AGGREGATION

In [8]:
query = """
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

df = pd.read_sql(query, engine)

df

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


JOIN

In [9]:
query = """
SELECT
    o.order_id,
    o.order_status,
    ROUND(
        SUM(oi.price + oi.freight_value),
        2
    ) AS order_value
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    o.order_id,
    o.order_status
ORDER BY order_value DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,order_id,order_status,order_value
0,03caa2c082116e1d31e67e9ae3700499,delivered,13664.08
1,736e1922ae60d0d6a89247b851902527,delivered,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,delivered,6929.31
3,fefacc66af859508bf1a7934eab1e97f,delivered,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,delivered,6726.66
5,2cc9089445046817a7539d90805e6e5a,delivered,6081.54
6,a96610ab360d42a2e5335a3998b4718a,delivered,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,canceled,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,delivered,4764.34
9,8dbc85d1447242f3b127dda390d56e19,delivered,4681.78


HAVING

In [10]:
query = """
SELECT
    seller_id,
    ROUND(
        SUM(price),
        2
    ) AS seller_revenue
FROM order_items
GROUP BY seller_id
HAVING SUM(price) > 10000
ORDER BY seller_revenue DESC;
"""

df = pd.read_sql(query, engine)

df

,seller_id,seller_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
...,...,...
287,1c5e4e49b9079480255b49d50aac1aa9,10173.11
288,7b1222c3624aa89b9558b50a2594188c,10103.52
289,5058e8c1e82653974541e83690655b4a,10022.82
290,0df3984f9dfb3d49ac6366acbd3bbb85,10021.49


SUBQUERY

In [11]:
query = """
SELECT
    seller_id,
    seller_revenue
FROM seller_features
WHERE seller_revenue > (
    SELECT AVG(seller_revenue)
    FROM seller_features
)
ORDER BY seller_revenue DESC;
"""

df = pd.read_sql(query, engine)

df

,seller_id,seller_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
...,...,...
623,b90e891671cffd9557f33a97dc523645,4449.00
624,2e3be8a987a30d7544dbbda6861cc14e,4430.84
625,d379f449f2a3b271bc01c0782020f705,4423.06
626,0ef83d7d83ed97cd2a0049ac8be5f88a,4421.01


CTE

In [20]:
query = """
WITH monthly_sales AS (
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, '%%Y-%%m') AS month,
        ROUND(
            SUM(oi.price + oi.freight_value),
            2
        ) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY month
)

SELECT
    month,
    revenue
FROM monthly_sales
ORDER BY month;
"""

df = pd.read_sql(query, engine)

df

,month,revenue
0,2016-09,354.75
1,2016-10,56808.84
2,2016-12,19.62
3,2017-01,137188.49
4,2017-02,286280.62
5,2017-03,432048.59
6,2017-04,412422.24
7,2017-05,586190.95
8,2017-06,502963.04
9,2017-07,584971.62


WINDOW FUNCTION

In [14]:
query = """
SELECT
    seller_id,
    seller_revenue,
    RANK() OVER (
        ORDER BY seller_revenue DESC
    ) AS revenue_rank
FROM seller_features
ORDER BY revenue_rank;
"""

df = pd.read_sql(query, engine)

df

,seller_id,seller_revenue,revenue_rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,5
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,3091
3091,702835e4b785b67a084280efca355756,7.60,3092
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,3093
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,3094


Revenue Contribution Using a Window Function

In [15]:
query = """
SELECT
    seller_id,
    seller_revenue,
    ROUND(
        seller_revenue /
        SUM(seller_revenue) OVER () * 100,
        2
    ) AS revenue_percentage
FROM seller_features
ORDER BY seller_revenue DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,seller_id,seller_revenue,revenue_percentage
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1.69
1,53243585a1d6dc2643021fd1853d8905,222776.05,1.64
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1.47
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,1.43
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,1.38
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,1.30
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1.18
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1.04
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,1.02
9,955fee9216a65b617aa5c0531780ce60,135171.70,0.99


Delivery Performance Analysis

In [16]:
query = """
SELECT
    delivery_status,
    COUNT(*) AS order_count,
    ROUND(AVG(delivery_days), 2) AS average_delivery_days,
    ROUND(AVG(delivery_delay_days), 2) AS average_delay_days
FROM order_features
GROUP BY delivery_status
ORDER BY order_count DESC;
"""

df = pd.read_sql(query, engine)

df

,delivery_status,order_count,average_delivery_days,average_delay_days
0,Early,88626,10.42,-13.71
1,Late,6535,33.49,10.62
2,Not Delivered,2988,NaN,NaN
3,On Time,1292,18.77,0.00


Product Category Performance

In [17]:
query = """
SELECT
    category_name,
    product_count,
    items_sold,
    order_count,
    category_revenue
FROM category_features
ORDER BY category_revenue DESC
LIMIT 15;
"""

df = pd.read_sql(query, engine)

df

,category_name,product_count,items_sold,order_count,category_revenue
0,health_beauty,2444,9670,8836,1258681.34
1,watches_gifts,1329,5991,5624,1205005.68
2,bed_bath_table,3029,11115,9417,1036988.68
3,sports_leisure,2867,8641,7720,988048.97
4,computers_accessories,1639,7827,6689,911954.32
5,furniture_decor,2657,8334,6449,729762.49
6,cool_stuff,789,3796,3632,635290.85
7,housewares,2335,6964,5884,632248.66
8,auto,1900,4235,3897,592720.11
9,garden_tools,753,4347,3518,485256.46


Top Customers by Spending

In [18]:
query = """
SELECT
    customer_unique_id,
    customer_order_count,
    customer_total_spending,
    average_order_value,
    repeat_customer_indicator
FROM customer_features
ORDER BY customer_total_spending DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df

,customer_unique_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer_indicator
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.08,0
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.82,1
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.88,0
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.31,0
4,459bef486812aa25204be022145caa62,1,6922.21,6922.21,0
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.66,0
6,4007669dec559734d6f53e029e360987,1,6081.54,6081.54,0
7,5d0a2980b292d049061542014e8960bf,1,4809.44,4809.44,0
8,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.34,0
9,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.78,0


Feature View Validation

In [19]:
required_views = [
    "order_features",
    "customer_features",
    "seller_features",
    "product_features",
    "category_features",
    "order_enriched_features"
]

with engine.connect() as connection:

    for view_name in required_views:

        result = connection.execute(
            text(f"SHOW FULL TABLES LIKE '{view_name}'")
        )

        rows = result.fetchall()

        if rows:
            print(f"✓ {view_name} exists")
        else:
            print(f"✗ {view_name} is missing")

✓ order_features exists
✓ customer_features exists
✓ seller_features exists
✓ product_features exists
✓ category_features exists
✓ order_enriched_features exists
